# Visualization of GPQR model

In [ ]:
import sys
import os
import warnings

import numpy as np
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

## 1D plot

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
y = pd.read_csv("../../_temp/v1/y.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_1D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
pred_gpqr = pd.read_csv("../../benchmarks/v1/gpqr.Xpred_1D.csv").pivot(
    index=["index", "batch", "quantile", "sample"],
    columns="target",
)

gpqr_levels = pred_gpqr.index.get_level_values("quantile").unique().sort_values()
mean_lowerq = (
    pred_gpqr["value"]
    .xs(gpqr_levels[0], level="quantile")
    .groupby(level="index")
    .mean()
)
mean_upperq = (
    pred_gpqr["value"]
    .xs(gpqr_levels[-1], level="quantile")
    .groupby(level="index")
    .mean()
)

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_y = y.loc[ok, "H"].to_numpy()

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        x_obs = this_X[ok]["gap_to_thickness_ratio"].to_numpy()
        y_obs = this_y[ok]
        x_values = np.sort(np.unique(x_obs))
        box_width = 0.6 * np.min(np.diff(x_values)) if len(x_values) > 1 else 0.1
        color = cmap(norm(ca))
        ax.boxplot(
            [y_obs[x_obs == x0] for x0 in x_values],
            positions=x_values,
            widths=box_width,
            whis=(2.5, 97.5),
            patch_artist=True,
            manage_ticks=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
            whiskerprops={"color": color},
            capprops={"color": color},
            medianprops={"color": color},
            flierprops={
                "marker": "o",
                "markerfacecolor": color,
                "markeredgecolor": color,
                "markersize": 2,
            },
        )

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            mean_lowerq.loc[prediction_index, "H"],
            mean_upperq.loc[prediction_index, "H"],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")
fig.suptitle("Quantiles")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_y = y.loc[ok, "phi_1"].to_numpy()

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].copy()
    this_Xpred["prediction_index"] = np.flatnonzero(ok_pred)

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        x_obs = this_X[ok]["gap_to_thickness_ratio"].to_numpy()
        y_obs = this_y[ok]
        x_values = np.sort(np.unique(x_obs))
        box_width = 0.6 * np.min(np.diff(x_values)) if len(x_values) > 1 else 0.1
        color = cmap(norm(ca))
        ax.boxplot(
            [y_obs[x_obs == x0] for x0 in x_values],
            positions=x_values,
            widths=box_width,
            whis=(2.5, 97.5),
            patch_artist=True,
            manage_ticks=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
            whiskerprops={"color": color},
            capprops={"color": color},
            medianprops={"color": color},
            flierprops={
                "marker": "o",
                "markerfacecolor": color,
                "markeredgecolor": color,
                "markersize": 2,
            },
        )

        prediction_index = this_Xpred.loc[ok_pred, "prediction_index"]
        ax.fill_between(
            this_Xpred.loc[ok_pred, "gap_to_thickness_ratio"],
            mean_lowerq.loc[prediction_index, "phi_1"],
            mean_upperq.loc[prediction_index, "phi_1"],
            facecolor=cmap(norm(ca)),
            edgecolor="none",
            alpha=0.3,
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel(r"$\phi$")
fig.suptitle("Quantiles")
plt.show()

## Distribution

In [ ]:
Xunique = pd.read_csv("../../_temp/v1/Xunique.csv", index_col=[0])
pred_gpqr = pd.read_csv("../../benchmarks/v1/gpqr.Xunique.csv").pivot(
    index=["index", "batch", "quantile", "sample"],
    columns="target",
)

In [ ]:
# Average posterior samples to obtain one predicted value per Xunique row and quantile.
posterior_mean_quantiles_all = (
    pred_gpqr["value"].groupby(level=["index", "quantile"]).mean()
)
quantile_probabilities = (
    posterior_mean_quantiles_all.index.get_level_values("quantile")
    .unique()
    .sort_values()
    .to_numpy()
)

# Match each observation to its Xunique row.
feature_columns = X.columns.tolist()
# GPQR prediction indices are zero-based Xunique row positions.
xunique_with_position = Xunique.reset_index(drop=True).assign(
    xunique_index=lambda frame: frame.index
)
observations = (
    X.reset_index(drop=True)
    .merge(
        xunique_with_position,
        on=feature_columns,
        how="left",
        validate="many_to_one",
    )
    .join(y.reset_index(drop=True))
)
if observations["xunique_index"].isna().any():
    raise ValueError("Some observations could not be matched to Xunique.")
observations["xunique_index"] = observations["xunique_index"].astype(int)

distribution_targets = pd.Index(["H", "phi_1"])
prediction_targets = distribution_targets.intersection(y.columns).intersection(
    posterior_mean_quantiles_all.columns
)
if prediction_targets.empty:
    raise ValueError("GPQR predictions do not contain any observed target columns.")

# At each predicted quantile value, compare the observed ECDF with its target probability.
# This probability-space score is unitless, so targets with larger numeric scales do not dominate.
ecdf_errors = {}
for target in prediction_targets:
    target_observations = observations[["xunique_index", target]].dropna()
    target_errors = {}
    for candidate_index, group in target_observations.groupby("xunique_index"):
        try:
            predicted_quantiles = (
                posterior_mean_quantiles_all[target]
                .xs(candidate_index, level="index")
                .reindex(quantile_probabilities)
                .to_numpy()
            )
        except KeyError:
            continue
        predicted_quantiles = np.maximum.accumulate(predicted_quantiles)
        observed_values = group[target].to_numpy()
        ecdf_at_predictions = (
            observed_values[:, None] <= predicted_quantiles[None, :]
        ).mean(axis=0)
        target_errors[candidate_index] = np.mean(
            (ecdf_at_predictions - quantile_probabilities) ** 2
        )
    ecdf_errors[target] = pd.Series(target_errors, dtype=float)

ecdf_scores = pd.DataFrame(ecdf_errors).dropna()
if ecdf_scores.empty:
    raise ValueError(
        "No Xunique row has observations and predictions for every target."
    )
mean_ecdf_error = ecdf_scores.mean(axis=1)
xunique_index = mean_ecdf_error.idxmin()
matching_y = observations.loc[observations["xunique_index"] == xunique_index, y.columns]

In [ ]:
# Read posterior-mean quantiles for the X value closest to the observed ECDF.
posterior_mean_quantiles = posterior_mean_quantiles_all.xs(xunique_index, level="index")

In [ ]:
# Compare the empirical CDF with the five posterior-mean quantile points.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

for ax, target, label in zip(axes, ["H", "phi_1"], ["H", r"$\phi_1$"]):
    quantile_values = (
        posterior_mean_quantiles[target].reindex(quantile_probabilities).to_numpy()
    )
    quantile_values = np.maximum.accumulate(quantile_values)
    observed_values = matching_y[target].dropna().to_numpy()
    observed_sorted = np.sort(observed_values)
    observed_cdf = np.arange(1, len(observed_sorted) + 1) / len(observed_sorted)

    ax.step(
        observed_sorted, observed_cdf, where="post", linewidth=2, label="Observed ECDF"
    )
    ax.plot(
        quantile_values,
        quantile_probabilities,
        marker="o",
        linewidth=2,
        color="C1",
        label="GPQR posterior mean",
    )
    ax.set_xlabel(label)
    ax.set_ylabel("Cumulative probability")
    ax.set_ylim(0, 1)
    ax.set_title(target)
    ax.grid(alpha=0.25)
    ax.legend()

plt.show()